# Methode B - LLM-based Transformation (Metadata)

**Concept:** The LLM receives the parsed Figma JSON and PrimeVue documentation as context and directly generates Vue 3 single-file components (SFCs) — without any explicit rules.

**Three Context Strategies (Variants):**

| Variant | Context                                     | Character                                       |
|---------|---------------------------------------------|-------------------------------------------------|
| **B1**  | No documentation                            | minimal, no api reference, error-prone          |
| **B2**  | Docs only for detected components (raw)     | focused, relevant, but noisy                    |
| **B3**  | Docs only for detected components (cleaned) | focused, relevant, clean, best expected results |

=> Variant with all raw docs not tested, because the large context would exceed the token limit for many files. The cleaned docs (B3) are expected to perform best, but B2 is also interesting to see the effect of cleaning.

**Prompt strategy:** Zero-shot and few-shot examples, no chain-of-thought.


In [63]:
import os
import re
import json
import time
import urllib.request
from pathlib import Path
from dotenv import load_dotenv
from collections import defaultdict

In [64]:
INPUT_DIR = 'dataset/figma-data/cleaned'
OUTPUT_DIR = 'dataset/storybook/src/stories'

DOCS_DIR_RAW = 'primevue/component-documentation/raw'
DOCS_DIR_CLEANED = 'primevue/component-documentation/cleaned'

# 'ZERO-SHOT' or 'FEW-SHOT'
PROMPT_STRATEGY = 'FEW-SHOT'

# TODO - use CLAUDE and Gemini too
API_URL = 'https://api.openai.com/v1/chat/completions'
API_MODEL = 'gpt-5.3-codex'
API_TEMPERATURE = 1.0       # 0.0–2.0 (1.0 = default; lower = more deterministic)
API_REASONING_EFFORT = 'medium'  # 'low' | 'medium' | 'high'

# Costs per token (codex-mini-latest: Input: $1.75 / Output: $14.00) (divided by one million)
INPUT_COSTS_PER_TOKEN  = 0.00000175
OUTPUT_COSTS_PER_TOKEN = 0.000014

load_dotenv(dotenv_path=Path('.env'))

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if OPENAI_API_KEY is None:
    raise ValueError('OPENAI_API_KEY not found in environment variables. Please set it in the .env file.')

## Load data and documentations

### 1.1 Load Raw Component-Documentations

In [65]:
DOCS_DIR_PATH = Path(DOCS_DIR_RAW)
RAW_DOCS: dict[str, str] = {}

for md_file in sorted(DOCS_DIR_PATH.glob('*.md')):
    RAW_DOCS[md_file.stem.lower()] = md_file.read_text(encoding='utf-8', errors='ignore')

print(f'Loaded Docs: {len(RAW_DOCS)}')
for name, content in RAW_DOCS.items():
    tokens_est = len(content) // 4
    print(f'  {name:20s}  ~{tokens_est:5d} Tokens')

TOTAL_TOKENS_B2 = sum(len(c) // 4 for c in RAW_DOCS.values())
print(f'\nTotal B2-Context: ~{TOTAL_TOKENS_B2:,} Tokens')

Loaded Docs: 25
  accordion             ~ 8091 Tokens
  avatar                ~ 2737 Tokens
  badge                 ~ 1860 Tokens
  breadcrumb            ~  988 Tokens
  button                ~14672 Tokens
  card                  ~ 1426 Tokens
  checkbox              ~ 4414 Tokens
  datatable             ~20664 Tokens
  datepicker            ~11753 Tokens
  dialog                ~ 9667 Tokens
  divider               ~ 4877 Tokens
  inputnumber           ~ 7011 Tokens
  inputtext             ~ 6664 Tokens
  menu                  ~ 3240 Tokens
  password              ~ 8082 Tokens
  popover               ~ 3859 Tokens
  progressbar           ~ 1173 Tokens
  radiobutton           ~ 4283 Tokens
  select                ~11278 Tokens
  skeleton              ~ 3134 Tokens
  slider                ~ 2688 Tokens
  tabs                  ~ 7255 Tokens
  tag                   ~ 1966 Tokens
  textarea              ~ 6287 Tokens
  toggleswitch          ~ 2693 Tokens

Total B2-Context: ~150,762 Tokens

### 1.2 Load Cleand Component-Documentations

In [66]:
DOCS_DIR_PATH = Path(DOCS_DIR_CLEANED)
CLEANED_DOCS: dict[str, str] = {}

for md_file in sorted(DOCS_DIR_PATH.glob('*.md')):
    CLEANED_DOCS[md_file.stem.lower()] = md_file.read_text(encoding='utf-8', errors='ignore')

print(f'Loaded Docs: {len(CLEANED_DOCS)}')
for name, content in CLEANED_DOCS.items():
    tokens_est = len(content) // 4
    print(f'  {name:20s}  ~{tokens_est:5d} Tokens')

TOTAL_TOKENS_B3 = sum(len(c) // 4 for c in CLEANED_DOCS.values())
print(f'\nTotal B3-Context: ~{TOTAL_TOKENS_B3:,} Tokens')

Loaded Docs: 25
  accordion             ~ 1811 Tokens
  avatar                ~  432 Tokens
  badge                 ~  283 Tokens
  breadcrumb            ~  169 Tokens
  button                ~ 3820 Tokens
  card                  ~  501 Tokens
  checkbox              ~  626 Tokens
  datatable             ~ 2480 Tokens
  datepicker            ~ 1755 Tokens
  dialog                ~ 1349 Tokens
  divider               ~  725 Tokens
  inputnumber           ~ 1634 Tokens
  inputtext             ~ 3649 Tokens
  menu                  ~  539 Tokens
  password              ~ 4501 Tokens
  popover               ~  761 Tokens
  progressbar           ~  272 Tokens
  radiobutton           ~  571 Tokens
  select                ~ 2065 Tokens
  skeleton              ~  438 Tokens
  slider                ~  487 Tokens
  tabs                  ~ 1641 Tokens
  tag                   ~  360 Tokens
  textarea              ~ 3549 Tokens
  toggleswitch          ~  587 Tokens

Total B3-Context: ~35,005 Tokens


### 1.3 Load Figma JSON Data

In [67]:
INPUT_DIR_PATH = Path(INPUT_DIR)
FIGMA_DATA: dict[str, dict] = {}

input_files = sorted(
    f for f in INPUT_DIR_PATH.rglob('*.json')
    if f.parent != INPUT_DIR_PATH
)

# For testing: only load one file per complexity level
#seen_complexities = set()
#input_files = []
#for f in all_input_files:
#    complexity = f.parent.name
#
#    if complexity not in seen_complexities:
#        seen_complexities.add(complexity)
#
#        input_files.append(f)

for input_file in input_files:
    print(f'Loading {input_file}...')

    with open(input_file, encoding='utf-8-sig') as f:
        key = f'{input_file.parent.name}-{input_file.stem}'.lower()
        FIGMA_DATA[key] = json.load(f)

print(f'\nLoaded Figma JSONs: {len(FIGMA_DATA)}')
for name, data in FIGMA_DATA.items():
    tokens_est = len(json.dumps(data)) // 4

    print(f'  {name:20s}  ~{tokens_est:5d} Tokens')

Loading dataset\figma-data\cleaned\hard\1.json...
Loading dataset\figma-data\cleaned\hard\10.json...
Loading dataset\figma-data\cleaned\hard\2.json...
Loading dataset\figma-data\cleaned\hard\3.json...
Loading dataset\figma-data\cleaned\hard\4.json...
Loading dataset\figma-data\cleaned\hard\5.json...
Loading dataset\figma-data\cleaned\hard\6.json...
Loading dataset\figma-data\cleaned\hard\7.json...
Loading dataset\figma-data\cleaned\hard\8.json...
Loading dataset\figma-data\cleaned\hard\9.json...
Loading dataset\figma-data\cleaned\medium\1.json...
Loading dataset\figma-data\cleaned\medium\10.json...
Loading dataset\figma-data\cleaned\medium\2.json...
Loading dataset\figma-data\cleaned\medium\3.json...
Loading dataset\figma-data\cleaned\medium\4.json...
Loading dataset\figma-data\cleaned\medium\5.json...
Loading dataset\figma-data\cleaned\medium\6.json...
Loading dataset\figma-data\cleaned\medium\7.json...
Loading dataset\figma-data\cleaned\medium\8.json...
Loading dataset\figma-data\cle

## 2. Detect PrimeVue-Components in Figma Data

In [68]:
KNOWN_COMPONENTS = set(CLEANED_DOCS.keys())

# Compound components that appear as FRAME
FRAME_COMPONENTS = {
    'card', 'dialog', 'tabs', 'datatable', 'select',
    'popover', 'breadcrumb', 'accordion',
}
# Aliases (Figma-Name → Doc-Name)
DOC_ALIASES = {
    'calendar': 'datepicker',
    'overlaybadge': 'badge',
}


def _normalize(name: str) -> str:
    return re.sub(r'[\s\-_]+', '', name or '').lower()


def detect_components(figma_node: dict, found: set | None = None) -> set[str]:
    """Collect all PrimeVue component names that appear in the Figma JSON"""
    if found is None:
        found = set()

    if not isinstance(figma_node, dict):
        return found

    name = figma_node.get('name', '')
    t = figma_node.get('type', '')

    if name.startswith('_'):
        return found  # interne Sub-Instance

    norm = _normalize(name)
    norm = DOC_ALIASES.get(norm, norm)

    if t in ('INSTANCE', 'FRAME') and norm in KNOWN_COMPONENTS:
        found.add(norm)

    for child in figma_node.get('children', []) or []:
        detect_components(child, found)

    return found


DETECTED_COMPONENTS: dict[str, set[str]] = {}

for key, data in FIGMA_DATA.items():
    detected = detect_components(data)
    DETECTED_COMPONENTS[key] = detected

    print(f'{key:20s}  → Detected: {", ".join(sorted(detected)) or "None"}')

print(f'\nTotal Unique Detected Components: {len(set.union(*DETECTED_COMPONENTS.values()))}')

hard-1                → Detected: button, checkbox, dialog, inputtext, select
hard-10               → Detected: button, datatable, popover
hard-2                → Detected: datatable
hard-3                → Detected: button, datepicker, inputnumber, inputtext, select
hard-4                → Detected: button, datatable, popover
hard-5                → Detected: button, dialog, tabs
hard-6                → Detected: inputtext, select
hard-7                → Detected: avatar, button, card, divider, tag
hard-8                → Detected: button, card, tabs
hard-9                → Detected: avatar, breadcrumb, button, datepicker, dialog, divider, inputtext, textarea
medium-1              → Detected: button, card, checkbox, inputtext, password
medium-10             → Detected: button, card, progressbar
medium-2              → Detected: tabs
medium-3              → Detected: accordion
medium-4              → Detected: breadcrumb, button, menu
medium-5              → Detected: button, card, inp

## 3. Prompt Building

### 3.1 Context-Builder

In [69]:
def build_context(strategy: str, figma_node: dict | None = None) -> tuple[str, list[str]]:
    """Returns (context_string, used_components)

    strategy: 'b1' | 'b2' | 'b3'
    figma_node: Required b2 and b3 for component recognition
    """

    if strategy == 'b1':
        return '', []  # No context for B1

    if figma_node is None:
        raise ValueError('Strategies requires figma_node for component recognition.')

    docs = RAW_DOCS if strategy == 'b2' else CLEANED_DOCS

    detected_components = detect_components(figma_node)

    context_parts = []
    used_components = []

    for comp in sorted(detected_components):
        if comp in docs:
            print(f'  Adding doc for component: {comp} ({len(docs[comp])//4} tokens)')

            context_parts.append(f'# {comp}\n\n{docs[comp]}')
            used_components.append(comp)

    return '\n\n'.join(context_parts), used_components # Only docs of detected components in raw or cleaned form, depending on strategy

### 3.2 Prompt Templates

In [70]:
SYSTEM_PROMPT_TEMPLATE_ZERO_SHOT = """You are an expert Vue 3 and PrimeVue developer.
Analyse the given Figma mockup JSON data and transform it into a complete, working Vue 3 Single File Component with PrimeVue 4 components. Use the provided PrimeVue documentation if given as reference for component usage and props.

STRICT REQUIREMENTS:
- Use PrimeVue 4 components exclusively for UI elements
- Use <script setup> syntax (no Options API)
- Import every PrimeVue component used: import Button from 'primevue/button'
- Use Tailwind CSS utility classes for layout and spacing
- Use ref() from Vue for all form/input state
- Map Figma Auto-Layout (HORIZONTAL/VERTICAL) to flex/flex-col
- Map itemSpacing to gap-*, padding values to p-*/px-*/py-*
- Output ONLY the Vue SFC — no explanation, no markdown fences, no prose
- Return exactly one complete Vue SFC, starting directly with <template> and ending with </script>
- If required details are missing or ambiguous, do not invent unsupported behavior; use the closest valid structural mapping supported by the Figma JSON
- Treat the transformation as incomplete until all eligible non-ignored nodes are represented in the output
- Before finalizing, verify that the SFC is syntactically valid, all used PrimeVue components are imported, and all form/input state uses ref()
- Assume PrimeVue Aura theme as baseline for styling; do not generate custom theme CSS unless explicitly required by Figma mockup JSON data

FIGMA JSON DATA STRUCTURE:
- type=INSTANCE, name=<component>: a PrimeVue component instance
- componentProperties: Figma design properties to map to PrimeVue props
- type=FRAME: layout container → <div> with Tailwind classes
- type=TEXT: standalone text → <span> or semantic element
- Nodes with name starting with '_' are internal sub-instances (ignore them)

PrimeVue DOCUMENTATION:
{context}"""

SYSTEM_PROMPT_TEMPLATE_FEW_SHOT = SYSTEM_PROMPT_TEMPLATE_ZERO_SHOT + '''

FEW-SHOT EXAMPLES for mapping Figma JSON to Vue 3 SFCs:

Example SIMPLE composite:
```json
{
  "type": "FRAME",
  "name": "Column [Simple Composite]",
  "layoutMode": "VERTICAL",
  "itemSpacing": 16.0,
  "paddingLeft": 24.0,
  "paddingRight": 24.0,
  "paddingTop": 24.0,
  "paddingBottom": 24.0,
  "children": [
    {
      "type": "FRAME",
      "name": "Row",
      "layoutMode": "HORIZONTAL",
      "itemSpacing": 16.0,
      "children": [
        {
          "type": "INSTANCE",
          "name": "avatar",
          "componentProperties": {
            "Text": "B",
            "Show Badge": false,
            "Size": "X-Large",
            "Type": "Label",
            "Circle": "True"
          }
        },
        {
          "type": "TEXT",
          "name": "Benutzername",
          "characters": "Benutzername"
        }
      ]
    },
    {
      "type": "INSTANCE",
      "name": "textarea",
      "componentProperties": {
        "Float Label": "Placeholder",
        "Show Text": true,
        "State": "Default",
        "Invalid": "False",
        "Disabled": "False",
        "Filled": "False",
        "Size": "Normal",
        "Ifta Label": "False",
        "Float Label": "False",
        "Float Label Variant": "N/A"
      }
    },
    {
      "type": "FRAME",
      "name": "Row",
      "layoutMode": "HORIZONTAL",
      "children": [
        {
          "type": "INSTANCE",
          "name": "checkbox",
          "componentProperties": {
            "Label": "Benachrichtigen",
            "Show Label": true,
            "Hover": "False",
            "Selected": "False",
            "Focus": "False",
            "Disabled": "False",
            "Filled": "False",
            "Size": "Normal"
          }
        },
        {
          "type": "INSTANCE",
          "name": "button",
          "componentProperties": {
            "Left Icon": "7:2160",
            "Icon": "7:2160",
            "Text": "Senden",
            "Show Right Icon": false,
            "Right Icon": "7:2160",
            "Show Left Icon": false,
            "Severity": "Primary",
            "State": "Idle",
            "Disabled": "False",
            "Icon Only": "False",
            "Raised": "False",
            "Rounded": "False",
            "Text": "False",
            "Outlined": "False",
            "Link": "False"
          }
        }
      ]
    }
  ]
}
```

```vue
<template>
  <div class="flex w-lg flex-col gap-4 p-6">
    <div class="flex items-center gap-4">
      <Avatar label="B" size="xlarge" shape="circle" />
      <span class="text-xl text-black">Benutzername</span>
    </div>
    <Textarea v-model="feedback" placeholder="Feedback eingeben..." />
    <div class="flex items-center justify-between">
      <div class="flex items-center gap-2">
        <Checkbox v-model="notification" input-id="notification" binary />
        <label for="notification">Benachrichtigen</label>
      </div>
      <Button label="Senden" severity="primary" class="w-fit" />
    </div>
  </div>
</template>

<script setup lang="ts">
  import { ref } from 'vue'
  import Avatar from 'primevue/avatar'
  import Button from 'primevue/button'
  import Textarea from 'primevue/textarea'
  import Checkbox from 'primevue/checkbox'

  const feedback = ref('')
  const notification = ref(false)
</script>
```

Example MEDIUM composite:
```json
{
  "type": "FRAME",
  "name": "Card [Medium Composite]",
  "layoutMode": "VERTICAL",
  "itemSpacing": 7.0,
  "children": [
    {
      "type": "FRAME",
      "name": "body",
      "layoutMode": "VERTICAL",
      "itemSpacing": 16.0,
      "paddingLeft": 24.0,
      "paddingRight": 24.0,
      "paddingTop": 24.0,
      "paddingBottom": 24.0,
      "children": [
        {
          "type": "FRAME",
          "name": "caption",
          "layoutMode": "VERTICAL",
          "itemSpacing": 7.0,
          "children": [
            {
              "type": "TEXT",
              "name": "Anmelden",
              "characters": "Anmelden"
            }
          ]
        },
        {
          "type": "FRAME",
          "name": "content",
          "layoutMode": "VERTICAL",
          "itemSpacing": 16.0,
          "children": [
            {
              "type": "INSTANCE",
              "name": "inputtext",
              "componentProperties": {
                "Float Label": "Placeholder",
                "Show Label": false,
                "Show Helper": false,
                "Helper Text": "Helper Text",
                "Show Right Icon": false,
                "Right Icon": "7:29",
                "Show Left Icon": false,
                "Left Icon": "7:29",
                "Label": "Label",
                "Show Text": true,
                "State": "Default",
                "Invalid": "False",
                "Disabled": "False",
                "Filled": "False",
                "Size": "Normal",
                "Ifta Label": "False",
                "Float Label": "False",
                "Float Label Variant": "N/A"
              }
            },
            {
              "type": "INSTANCE",
              "name": "password",
              "componentProperties": {
                "State": "Selected",
                "Toggle Mask": "True",
                "Password Visible": "False"
              }
            },
            {
              "type": "FRAME",
              "name": "Frame 1",
              "layoutMode": "HORIZONTAL",
              "children": [
                {
                  "type": "INSTANCE",
                  "name": "tag",
                  "componentProperties": {
                    "Icon": "7:7029",
                    "Text": "Beliebt",
                    "Show Icon": false,
                    "Severity": "Info",
                    "Rounded": "False"
                  }
                },
                {
                  "type": "INSTANCE",
                  "name": "progressbar",
                  "componentProperties": {
                    "Text": "",
                    "Type": "Basic",
                    "Value": "True"
                  }
                }
              ]
            }
          ]
        },
        {
          "type": "FRAME",
          "name": "footer",
          "layoutMode": "HORIZONTAL",
          "itemSpacing": 7.0,
          "children": [
            {
              "type": "INSTANCE",
              "name": "button",
              "componentProperties": {
                "Left Icon": "7:2160",
                "Icon": "7:2160",
                "Text": "Jetzt starten",
                "Show Right Icon": false,
                "Right Icon": "7:2160",
                "Show Left Icon": false,
                "Severity": "Primary",
                "State": "Idle",
                "Disabled": "False",
                "Icon Only": "False",
                "Raised": "False",
                "Rounded": "False",
                "Text": "False",
                "Outlined": "False",
                "Link": "False"
              }
            }
          ]
        }
      ]
    }
  ]
}
```

```vue
<template>
  <Card
    :pt="{
      root: 'w-md p-8 gap-6',
      body: 'flex flex-col gap-4 !p-0',
      content: 'flex flex-col gap-4',
      footer: 'mt-2',
    }"
  >
    <template #header>
      <h1 class="text-lg font-medium">Anmelden</h1>
    </template>
    <template #content>
      <InputText v-model="email" type="email" placeholder="E-Mail-Adresse" input-id="email-input" />
      <Password v-model="password" input-id="password-input" toggle-mask input-class="w-full" />
      <div class="flex items-center justify-between">
        <Badge value="Beliebt" severity="info" />
        <ProgressBar :value="50" :show-value="false" class="!h-1 w-[84px]" />
      </div>
    </template>
    <template #footer>
      <Button label="Jetzt starten" severity="primary" class="w-full" />
    </template>
  </Card>
</template>

<script setup lang="ts">
  import { ref } from 'vue'
  import Card from 'primevue/card'
  import Badge from 'primevue/badge'
  import Button from 'primevue/button'
  import Password from 'primevue/password'
  import InputText from 'primevue/inputtext'
  import ProgressBar from 'primevue/progressbar'

  const email = ref('')
  const password = ref('password')
</script>
```

Example HARD composite:
```json
{
  "type": "FRAME",
  "name": "Page [Hard composite]",
  "children": [
    {
      "type": "FRAME",
      "name": "datatable",
      "layoutMode": "VERTICAL",
      "children": [
        {
          "type": "FRAME",
          "name": "thead",
          "layoutMode": "HORIZONTAL",
          "children": [
            {
              "type": "TEXT",
              "name": "Projekt",
              "characters": "Projekt"
            },
            {
              "type": "TEXT",
              "name": "Status",
              "characters": "Status"
            },
            {
              "type": "TEXT",
              "name": "Aktionen",
              "characters": "Aktionen"
            }
          ]
        },
        {
          "type": "FRAME",
          "name": "tbody",
          "layoutMode": "VERTICAL",
          "children": [
            {
              "type": "TEXT",
              "name": "Content",
              "characters": "Webseite Relaunch"
            },
            {
              "type": "INSTANCE",
              "name": "tag",
              "componentProperties": {
                "Icon": "7:7029",
                "Text": "Aktiv",
                "Show Icon": false,
                "Severity": "Primary",
                "Rounded": "False"
              }
            },
            {
              "type": "INSTANCE",
              "name": "button",
              "componentProperties": {
                "Right Icon": "7:2160",
                "Left Icon": "7:2160",
                "Show Left Icon": false,
                "Text": "Show",
                "Show Right Icon": false,
                "Icon": "32:3127",
                "Severity": "Primary",
                "State": "Active",
                "Disabled": "False",
                "Icon Only": "True",
                "Raised": "False",
                "Rounded": "False",
                "Text": "True",
                "Outlined": "False",
                "Link": "False"
              }
            }
          ]
        }
      ]
    },
    {
      "type": "FRAME",
      "name": "popover",
      "layoutMode": "VERTICAL",
      "children": [
        {
          "type": "FRAME",
          "name": "popover",
          "layoutMode": "VERTICAL",
          "itemSpacing": 14.0,
          "children": [
            {
              "type": "FRAME",
              "name": "popover",
              "layoutMode": "VERTICAL",
              "children": [
                {
                  "type": "FRAME",
                  "name": "content",
                  "layoutMode": "VERTICAL",
                  "itemSpacing": 7.0,
                  "paddingLeft": 10.5,
                  "paddingRight": 10.5,
                  "paddingTop": 10.5,
                  "paddingBottom": 10.5,
                  "children": [
                    {
                      "type": "FRAME",
                      "name": "col",
                      "layoutMode": "VERTICAL",
                      "itemSpacing": 8.0,
                      "children": [
                        {
                          "type": "INSTANCE",
                          "name": "button",
                          "componentProperties": {
                            "Icon": "7:2160",
                            "Right Icon": "7:2160",
                            "Text": "Bearbeiten",
                            "Left Icon": "32:3139",
                            "Show Right Icon": false,
                            "Show Left Icon": true,
                            "Severity": "Secondary",
                            "State": "Idle",
                            "Disabled": "False",
                            "Icon Only": "False",
                            "Raised": "False",
                            "Rounded": "False",
                            "Text": "False",
                            "Outlined": "True",
                            "Link": "False"
                          }
                        },
                        {
                          "type": "INSTANCE",
                          "name": "button",
                          "componentProperties": {
                            "Icon": "7:2160",
                            "Right Icon": "7:2160",
                            "Text": "Löschen",
                            "Left Icon": "32:3133",
                            "Show Right Icon": false,
                            "Show Left Icon": true,
                            "Severity": "Secondary",
                            "State": "Idle",
                            "Disabled": "False",
                            "Icon Only": "False",
                            "Raised": "False",
                            "Rounded": "False",
                            "Text": "False",
                            "Outlined": "True",
                            "Link": "False"
                          }
                        }
                      ]
                    }
                  ]
                }
              ]
            }
          ]
        }
      ]
    },
    {
      "type": "FRAME",
      "name": "screen",
      "layoutMode": "VERTICAL",
      "paddingTop": 280.0,
      "paddingBottom": 280.0,
      "children": [
        {
          "type": "FRAME",
          "name": "dialog",
          "layoutMode": "VERTICAL",
          "children": [
            {
              "type": "FRAME",
              "name": "header",
              "layoutMode": "HORIZONTAL",
              "paddingLeft": 17.5,
              "paddingRight": 17.5,
              "paddingTop": 17.5,
              "paddingBottom": 17.5,
              "children": [
                {
                  "type": "TEXT",
                  "name": "Projekt bearbeiten",
                  "characters": "Projekt bearbeiten"
                },
                {
                  "type": "FRAME",
                  "name": "actions",
                  "layoutMode": "VERTICAL",
                  "itemSpacing": 7.0,
                  "children": [
                    {
                      "type": "INSTANCE",
                      "name": "button",
                      "componentProperties": {
                        "Text": "Button",
                        "Show Right Icon": true,
                        "Right Icon": "7:2160",
                        "Left Icon": "7:2160",
                        "Show Left Icon": true,
                        "Icon": "9:255",
                        "Severity": "Secondary",
                        "State": "Idle",
                        "Disabled": "False",
                        "Icon Only": "True",
                        "Raised": "False",
                        "Rounded": "False",
                        "Text": "True",
                        "Outlined": "False",
                        "Link": "False"
                      }
                    }
                  ]
                }
              ]
            },
            {
              "type": "FRAME",
              "name": "content",
              "layoutMode": "HORIZONTAL",
              "itemSpacing": 7.0,
              "paddingLeft": 17.5,
              "paddingRight": 17.5,
              "paddingBottom": 17.5,
              "children": [
                {
                  "type": "INSTANCE",
                  "name": "inputtext",
                  "componentProperties": {
                    "Float Label": "Placeholder",
                    "Show Label": true,
                    "Show Helper": false,
                    "Helper Text": "Helper Text",
                    "Show Right Icon": false,
                    "Right Icon": "7:29",
                    "Show Left Icon": false,
                    "Left Icon": "7:29",
                    "Label": "Name",
                    "Show Text": true,
                    "State": "Default",
                    "Invalid": "False",
                    "Disabled": "False",
                    "Filled": "False",
                    "Size": "Normal",
                    "Ifta Label": "False",
                    "Float Label": "False",
                    "Float Label Variant": "N/A"
                  }
                }
              ]
            },
            {
              "type": "FRAME",
              "name": "footer",
              "layoutMode": "HORIZONTAL",
              "itemSpacing": 7.0,
              "paddingLeft": 17.5,
              "paddingRight": 17.5,
              "paddingBottom": 17.5,
              "children": [
                {
                  "type": "INSTANCE",
                  "name": "button",
                  "componentProperties": {
                    "Show Right Icon": false,
                    "Right Icon": "7:2160",
                    "Left Icon": "7:2160",
                    "Text": "Abbrechen",
                    "Icon": "7:2160",
                    "Show Left Icon": false,
                    "Severity": "Secondary",
                    "State": "Idle",
                    "Disabled": "False",
                    "Icon Only": "False",
                    "Raised": "False",
                    "Rounded": "False",
                    "Text": "False",
                    "Outlined": "False",
                    "Link": "False"
                  }
                },
                {
                  "type": "INSTANCE",
                  "name": "button",
                  "componentProperties": {
                    "Left Icon": "7:2160",
                    "Icon": "7:2160",
                    "Text": "Speichern",
                    "Show Right Icon": false,
                    "Right Icon": "7:2160",
                    "Show Left Icon": false,
                    "Severity": "Primary",
                    "State": "Idle",
                    "Disabled": "False",
                    "Icon Only": "False",
                    "Raised": "False",
                    "Rounded": "False",
                    "Text": "False",
                    "Outlined": "False",
                    "Link": "False"
                  }
                }
              ]
            }
          ]
        }
      ]
    }
  ]
}
```

```vue
<template>
  <DataTable :value="projects">
    <Column field="name" header="Name" />
    <Column field="status" header="Status">
      <template #body="{ data }">
        <Tag :value="data.status" :severity="getStatusTagSeverity(data.status)" />
      </template>
    </Column>
    <Column header="Aktionen" header-class="w-24" body-class="w-24 flex justify-center">
      <template #body>
        <Button
          icon="pi pi-ellipsis-h"
          severity="secondary"
          aria-haspopup="true"
          aria-controls="actions-menu"
          @click="actionsMenu?.toggle"
        />
      </template>
    </Column>
  </DataTable>
  <Menu
    ref="actions-menu"
    id="actions-menu"
    :model="actionOptions"
    popup
    :pt="{
      list: 'flex flex-col !gap-2 !p-2.5',
    }"
  >
    <template #item="{ item }">
      <Button
        :label="item.label"
        :icon="item.icon"
        severity="secondary"
        outlined
        class="w-full !justify-start"
      />
    </template>
  </Menu>
  <Dialog
    v-model:visible="isEditProjektDialogVisible"
    header="Projekt bearbeiten"
    modal
    :pt="{
      root: 'w-full max-w-md',
      content: 'flex flex-col !gap-4',
    }"
  >
    <div class="flex flex-col gap-2">
      <label for="name-input" class="text-sm">Name</label>
      <InputText v-model="name" type="text" input-id="name-input" />
    </div>
    <template #footer>
      <Button label="Abbrechen" severity="secondary" />
      <Button label="Speichern" severity="primary" />
    </template>
  </Dialog>
</template>

<script setup lang="ts">
  import { ref, useTemplateRef } from 'vue'
  import Tag from 'primevue/tag'
  import Column from 'primevue/column'
  import DataTable from 'primevue/datatable'
  import Button from 'primevue/button'
  import Menu from 'primevue/menu'
  import Dialog from 'primevue/dialog'
  import InputText from 'primevue/inputtext'

  const projects = ref([
    {
      name: 'Webseite Relaunch',
      status: 'Aktiv',
    },
  ])

  const isEditProjektDialogVisible = ref(true)
  const name = ref('Webseite Relaunch')

  const actionsMenu = useTemplateRef('actions-menu')
  const actionOptions = [
    {
      label: 'Bearbeiten',
      icon: 'pi pi-pen-to-square',
      command: () => (isEditProjektDialogVisible.value = true),
    },
    {
      label: 'Löschen',
      icon: 'pi pi-trash',
    },
  ]

  function getStatusTagSeverity(status: string) {
    switch (status) {
      case 'Aktiv':
        return 'success'
      case 'In Prüfung':
        return 'warn'
      case 'Abgeschlossen':
        return 'info'
      case 'Gestoppt':
        return 'danger'
    }
  }
</script>
```
'''

USER_PROMPT_TEMPLATE = """Transform the following Figma mockup JSON data into a Vue 3 Single File Component using PrimeVue components.

Figma mockup JSON data:
```json
{figma_json}
```"""

### 3.3 Prompt Builder

In [71]:
def _remove_outer_figma_frame(figma_root: dict) -> dict:
    """If root is a FRAME with one child, unwrap it to reduce nesting noise."""
    if figma_root.get('type') == 'FRAME' and len(figma_root.get('children', [])) == 1:
        return figma_root['children'][0]

    return figma_root


def build_prompts(figma_root: dict, strategy: str) -> tuple[str, str, list[str], int]:
    """Creates system and user prompts

    Returns: (system_prompt, user_prompt, used_components, context_tokens)
    """
    figma_root = _remove_outer_figma_frame(figma_root)

    context, used_components = build_context(strategy, figma_root)
    context_tokens = len(context) // 4

    prompt_template = (
            SYSTEM_PROMPT_TEMPLATE_FEW_SHOT
            if PROMPT_STRATEGY == 'FEW-SHOT'
            else SYSTEM_PROMPT_TEMPLATE_ZERO_SHOT
        )


    context_text = context if context else '(No documentation provided use pretrained knowledge)'
    # Use a plain replace to avoid KeyError from JSON braces in few-shot examples.
    system_prompt = prompt_template.replace('{context}', context_text)

    user_prompt = USER_PROMPT_TEMPLATE.format(
        figma_json=json.dumps(figma_root, ensure_ascii=False, indent=2)
    )

    return system_prompt, user_prompt, used_components, context_tokens

## 4. LLM Interaction

In [72]:
def call_llm(system_prompt: str, user_prompt: str, strategy: str, key: str) -> dict:
    """Calls the OpenAI Responses API with text input

    Returns: {
        'content': str,
        'input_tokens': int,
        'output_tokens': int,
        'stop_reason': str,
        'duration': float,
    }
    """
    metadata = {
        'strategy':   strategy,
        'mockup_key': key,
    }

    payload = json.dumps( {
        'model': API_MODEL,
        'metadata': metadata,
        'temperature': API_TEMPERATURE,
        'reasoning': {'effort': API_REASONING_EFFORT},
        'input': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt},
        ],
    }).encode('utf-8')

    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {OPENAI_API_KEY}',
    }

    req = urllib.request.Request(
        API_URL,
        data=payload,
        headers=headers,
        method='POST'
    )

    start_time = time.time()

    try:
        with urllib.request.urlopen(req) as resp:
            resp_data = json.loads(resp.read().decode('utf-8'))
    except Exception as e:
        error_text = e.read().decode('utf-8', errors='ignore')

        try:
            error_body = json.loads(error_text)

            detail = error_body.get('error', {}).get('message', error_text)
        except json.JSONDecodeError:
            detail = error_text or str(e)

        raise RuntimeError(f'OpenAI API Fehler {e.code}: {detail}') from e

    end_time = time.time()

    usage = resp_data.get('usage', {})

    return {
        'content':       resp_data['output'][0]['content'][0]['text'],
        'input_tokens':  usage.get('input_tokens', 0),
        'output_tokens': usage.get('output_tokens', 0),
        'stop_reason':   resp_data['output'][0].get('status', ''),
        'duration':      end_time - start_time,
    }

In [73]:
def extract_sfc(raw: str) -> str:
    """Extracts the Vue SFC code from the LLM response

    Handles three cases:

    1. Direct SFC output (starts with <template>)
    2. Code block with language annotation (```vue ... ```)
    3. Generic code block (``` ... ```)
    """
    # Case 2: ```vue ... ```
    m = re.search(r'```vue\s*\n(.+?)```', raw, re.DOTALL)
    if m:
        return m.group(1).strip()

    # Case 3: ``` ... ```
    m = re.search(r'```\s*\n(.+?)```', raw, re.DOTALL)
    if m:
        candidate = m.group(1).strip()

        if '<template>' in candidate:
            return candidate

    # Case 1: Direct SFC
    if '<template>' in raw:
        start = raw.index('<template>')

        return raw[start:].strip()

    # Fallback: Return the raw response with a comment
    return f'<!-- SFC-Extraktion failed -->\n<!-- RAW:\n{raw[:500]}\n-->'

## 5. Record metrics

In [75]:
_metrics_b: dict = {}

def _reset_metrics_b():
    global _metrics_b
    _metrics_b = {
        'input_tokens':   0,
        'output_tokens':  0,
        'context_tokens': 0,
        'context_components': 0,
        'stop_reason':    '',
        'duration': 0,
        'parse_ok':       False,
    }


def _ast_depth_approx(sfc: str) -> int:
    """Estimates the maximum template depth by counting tags"""
    depth, max_depth = 0, 0
    in_template = False

    for line in sfc.splitlines():
        if '<template>' in line:
            in_template = True

        if not in_template:
            continue

        depth += line.count('<') - line.count('</') - line.count('/>')
        max_depth = max(max_depth, depth)

    return max(0, max_depth)

## 6. Main-Transformation for one Figma JSON with a given strategy

In [76]:
def generate_sfc_b(figma_root: dict, strategy: str, key: str) -> str:
    """Transforms a Figma mockup using the specified LLM strategy.

    strategy: 'b1' | 'b2' | 'b3'
    Returns: Vue-3-SFC als String
    """
    _reset_metrics_b()

    system_prompt, user_prompt, used_components, context_tokens = \
        build_prompts(figma_root, strategy)

    print(f'Detected Components: {used_components} → Context Tokens: {context_tokens}')

    _metrics_b['context_tokens']     = context_tokens
    _metrics_b['context_components'] = len(used_components)

    response = call_llm(system_prompt, user_prompt, strategy, key)

    _metrics_b['input_tokens']  = response['input_tokens']
    _metrics_b['output_tokens'] = response['output_tokens']
    _metrics_b['stop_reason']   = response['stop_reason']
    _metrics_b['duration']      = response['duration']

    sfc = extract_sfc(response['content'])

    _metrics_b['parse_ok'] = '<template>' in sfc and '<script' in sfc

    return sfc

## 7. Pipeline for all Figma JSONs and all strategies

In [77]:
STRATEGIES = ['b1', 'b2', 'b3']
OUTPUT_PATH = Path(OUTPUT_DIR)

all_results: list[dict] = []

print(f'Input-Files: {len(FIGMA_DATA)} Strategies: {STRATEGIES}\n')


def _complexity_from_key(key: str) -> str:
    """Extract complexity prefix from keys like 'medium-8' or 'simple-5'."""
    return key.split('-', 1)[0] if '-' in key else 'unknown'


for key, figma_root in FIGMA_DATA.items():
    complexity = _complexity_from_key(key)

    print(f' Processing {key}...\n{"="*60}')

    for strategy in STRATEGIES:

        try:
            print(f' Strategy: {strategy.upper()}')

            sfc = generate_sfc_b(figma_root, strategy, key)

            out_path = OUTPUT_PATH / complexity / f'{key.split("-", 1)[1]}-{strategy}-{PROMPT_STRATEGY.lower()}.vue'
            out_path.parent.mkdir(parents=True, exist_ok=True)
            out_path.write_text(sfc, encoding='utf-8')

            m = dict(_metrics_b)
            result = {
                'input':               key,
                'output':              out_path.name,
                'complexity':          complexity,
                'strategy':            strategy,
                'duration_ms':         round(m['duration'] * 1000, 4),
                'sfc_bytes':           len(sfc),
                'sfc_lines':           sfc.count('\n') + 1,
                'ast_depth_approx':    _ast_depth_approx(sfc),
                'parse_ok':            m['parse_ok'],
                'input_tokens':        m['input_tokens'],
                'output_tokens':       m['output_tokens'],
                'context_tokens':      m['context_tokens'],
                'context_components':  m['context_components'],
                'stop_reason':         m['stop_reason'],
                'cost_usd':            round(
                    m['input_tokens'] * INPUT_COSTS_PER_TOKEN + m['output_tokens'] * OUTPUT_COSTS_PER_TOKEN,
                    6
                 ),
                'error':               None,
            }

            print(f'  OK  {key:25s} [{strategy}]  '
                  f'in={m["input_tokens"]:5d}tok  '
                  f'out={m["output_tokens"]:4d}tok  '
                  f'${result["cost_usd"]:.4f}  '
                  f'{m["duration"]:6.0f}ms')

        except Exception as e:
            print(f'  ERROR {key:25s} [{strategy}]  {str(e)}')

            result = {
                'input': key,
                'output': None,
                'complexity': complexity,
                'strategy': strategy,
                'error': str(e),
                **{k: None for k in [
                    'duration_ms','sfc_bytes','sfc_lines','ast_depth_approx',
                    'parse_ok','input_tokens','output_tokens','context_tokens',
                    'context_components','stop_reason','cost_usd'
                ]},
            }

        all_results.append(result)


print(f'\nCompleted {len(all_results)} transformed')

Input-Files: 30 Strategies: ['b1', 'b2', 'b3']

 Processing hard-1...
 Strategy: B1
Detected Components: [] → Context Tokens: 0
  OK  hard-1                    [b1]  in= 9338tok  out= 492tok  $0.0232       6ms
 Strategy: B2
  Adding doc for component: button (14672 tokens)
  Adding doc for component: checkbox (4414 tokens)
  Adding doc for component: dialog (9667 tokens)
  Adding doc for component: inputtext (6664 tokens)
  Adding doc for component: select (11278 tokens)
Detected Components: ['button', 'checkbox', 'dialog', 'inputtext', 'select'] → Context Tokens: 46713
  OK  hard-1                    [b2]  in=55848tok  out= 481tok  $0.1045       7ms
 Strategy: B3
  Adding doc for component: button (3820 tokens)
  Adding doc for component: checkbox (626 tokens)
  Adding doc for component: dialog (1349 tokens)
  Adding doc for component: inputtext (3649 tokens)
  Adding doc for component: select (2065 tokens)
Detected Components: ['button', 'checkbox', 'dialog', 'inputtext', 'select'] →

## 8. Save results to CSV

In [78]:
def _avg(vals):
    clean = [v for v in vals if v is not None]

    return round(sum(clean) / len(clean), 4) if clean else None


# Aggregation: per Strategy
by_strategy = defaultdict(list)
for r in all_results:
    by_strategy[r['strategy']].append(r)

per_strategy = {}
for s, items in by_strategy.items():
    ok_items = [i for i in items if not i['error']]
    per_strategy[s] = {
        'count':                  len(items),
        'errors':                 len(items) - len(ok_items),
        'parse_ok_rate':          _avg([i['parse_ok'] for i in ok_items]),
        'avg_duration_ms':        _avg([i['duration_ms'] for i in ok_items]),
        'avg_input_tokens':       _avg([i['input_tokens'] for i in ok_items]),
        'avg_output_tokens':      _avg([i['output_tokens'] for i in ok_items]),
        'avg_context_tokens':     _avg([i['context_tokens'] for i in ok_items]),
        'total_cost_usd':         round(sum(i['cost_usd'] or 0 for i in ok_items), 4),
        'avg_cost_usd_per_file':  _avg([i['cost_usd'] for i in ok_items]),
    }

# Aggregation: per Strategie × Complexity
by_strat_complexity = defaultdict(lambda: defaultdict(list))
for r in all_results:
    if not r['error']:
        by_strat_complexity[r['strategy']][r['complexity']].append(r)

per_strategy_complexity = {}
for s, levels in by_strat_complexity.items():
    per_strategy_complexity[s] = {
        lvl: {
            'count':              len(items),
            'avg_cost_usd':       _avg([i['cost_usd'] for i in items]),
            'avg_input_tokens':   _avg([i['input_tokens'] for i in items]),
            'avg_output_tokens':  _avg([i['output_tokens'] for i in items]),
            'avg_duration_ms':    _avg([i['duration_ms'] for i in items]),
            'parse_ok_rate':      _avg([i['parse_ok'] for i in items]),
        }
        for lvl, items in levels.items()
    }

metrics_report = {
    'method': 'B',
    'model':  API_MODEL,
    'prompt_strategy': PROMPT_STRATEGY.lower(),
    'strategies_run':  STRATEGIES,
    'per_strategy':    per_strategy,
    'per_strategy_complexity': per_strategy_complexity,
    'files':           all_results,
}

report_path = Path('reports') / f'metrics_report_b_{PROMPT_STRATEGY.lower()}.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_report, f, indent=4, ensure_ascii=False)

print(f'Transformation report saved to: {report_path}')

Transformation report saved to: reports\metrics_report_b_few-shot.json
